In [ ]:
# 1. INSTALL LIBRARIES
!pip install tensorflow opencv-python matplotlib scikit-learn

In [ ]:
# 2. IMPORT LIBRARIES
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [ ]:
# 3. DATASET PATH

# Example folder structure:
# Dataset/
#    Mild/
#    Moderate/
#    No_DR/
#    Proliferate_DR/
#    Severe/

DATASET_PATH = "/content/Dataset" 

In [ ]:
# 4. LOAD IMAGES

IMG_SIZE = 128

images = []
labels = []

classes = os.listdir(DATASET_PATH)

for label in classes:

    folder_path = os.path.join(DATASET_PATH, label)

    for img_name in os.listdir(folder_path):

        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)

        if img is not None:

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            images.append(img)
            labels.append(label)

print("Total Images:", len(images))

In [ ]:
# 5. PREPROCESSING

X = np.array(images)
Y = np.array(labels)

X = X / 255.0

encoder = LabelEncoder()

Y_encoded = encoder.fit_transform(Y)

Y_categorical = to_categorical(Y_encoded)

print(X.shape)
print(Y_categorical.shape)

In [ ]:
# 6. TRAIN TEST SPLIT

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y_categorical,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
# 7. BUILD CNN MODEL

cnn_model = Sequential()

cnn_model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(128,128,3)
    )
)

cnn_model.add(MaxPooling2D(pool_size=(2,2)))

cnn_model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

cnn_model.add(MaxPooling2D(pool_size=(2,2)))

cnn_model.add(
    Conv2D(
        128,
        (3,3),
        activation='relu'
    )
)

cnn_model.add(MaxPooling2D(pool_size=(2,2)))

cnn_model.add(Flatten())

cnn_model.add(Dense(128, activation='relu'))

cnn_model.add(Dropout(0.5))

cnn_model.add(
    Dense(
        Y_categorical.shape[1],
        activation='softmax'
    )
)

In [ ]:
# 8. COMPILE CNN MODEL

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 9. TRAIN CNN MODEL

history = cnn_model.fit(
    X_train,
    Y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, Y_test)
)

In [ ]:
# 10. EVALUATE CNN MODEL

loss, accuracy = cnn_model.evaluate(X_test, Y_test)

print("CNN Accuracy:", accuracy)

In [ ]:
# 11. ACCURACY GRAPH

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(["Train", "Validation"])

plt.show()

In [ ]:
# 12. FLATTEN DATA FOR ANN

X_ann = X.reshape(X.shape[0], -1)

print(X_ann.shape)

In [ ]:
# 13. TRAIN TEST SPLIT FOR ANN

X_train_ann, X_test_ann, Y_train_ann, Y_test_ann = train_test_split(
    X_ann,
    Y_categorical,
    test_size=0.2,
    random_state=42
)

In [ ]:
# 14. BUILD ANN MODEL

ann_model = Sequential()

ann_model.add(
    Dense(
        512,
        activation='relu',
        input_shape=(X_train_ann.shape[1],)
    )
)

ann_model.add(Dropout(0.5))

ann_model.add(Dense(256, activation='relu'))

ann_model.add(Dropout(0.5))

ann_model.add(
    Dense(
        Y_categorical.shape[1],
        activation='softmax'
    )
)

In [ ]:
# 15. COMPILE ANN MODEL

ann_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 16. TRAIN ANN MODEL

history_ann = ann_model.fit(
    X_train_ann,
    Y_train_ann,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_ann, Y_test_ann)
)

In [ ]:
# 17. EVALUATE ANN MODEL

loss_ann, accuracy_ann = ann_model.evaluate(
    X_test_ann,
    Y_test_ann
)

print("ANN Accuracy:", accuracy_ann)

In [ ]:
# 18. SAVE MODELS

cnn_model.save("diabetic_retinopathy_cnn.h5")
ann_model.save("diabetic_retinopathy_ann.h5")

print("Models Saved Successfully")